# Create British Heart Foundation Awards (GRANT PATTERN, multi-era PDF archive)

British Heart Foundation (BHF) is the UK's largest independent funder of
cardiovascular research (~£100M/yr in new awards). BHF publishes **no API,
CSV export, or per-grant landing pages** — only one **annual PDF report** of
"Research Grant Awards" per financial year (1 April -> 31 March), back to
2004/2005, linked from
`bhf.org.uk/for-professionals/information-for-researchers/previous-awards`.

The source script `scripts/local/bhf_to_s3.py` parses all ~20 annual PDFs by
**word x-coordinate** (PyMuPDF), recovering the implicit
`Reference number | Name | Institution | Grant title | Total` table grouped
under scheme/committee headings. It is the awards pipeline's first
**multi-era PDF archive** ingest: the layout drifts across the 20-year span
(modern header-columnar, mid headerless-columnar, old stacked two-column), and
the parser auto-adapts per PDF. See the script docstring for the full method.

**Awarding body:** British Heart Foundation — F4320319992 (GB). Path A
(F4320* Crossref-registered funder, present in `openalex.common.funder`);
canonical ror_id/doi come from the dim in Step 1.6 below.

**Schema choices / known limitations:**
- One row per award (the reference number is the source-authoritative grant id).
- `currency = 'GBP'` hardcoded (single-country funder); ~99.5% of awards
  publish a `£` amount.
- `funder_scheme` = the captured committee/scheme heading (e.g.
  `'Project Grants'`, `'Clinical Research Training Fellowships'`);
  `funding_type` derived from it (Fellowship->fellowship, Studentship/PhD->
  training, else research).
- **PI names are initials-only** (`Dr H F Jorgensen` -> given `'H F'`, family
  `'Jorgensen'`) and **no ORCIDs** are published -> `lead_investigator.orcid`
  is NULL. `affiliation.name` = the published institution string (UK
  institutions, high ROR coverage); `country = 'GB'` except International Awards.
- **No abstracts** are published -> `description` is NULL.
- **No exact per-grant dates** are published. The report states only the
  financial year an award was *made* in, so `start_year` = the FY start year
  and `start_date` is left NULL (no false day/month precision). `end_year` is
  derived from the published duration when present; `end_date` NULL.

**Prerequisites:** run `scripts/local/bhf_to_s3.py` first to download + parse
the PDFs and upload the parquet to S3.

**Data source:** https://www.bhf.org.uk/for-professionals/information-for-researchers/previous-awards
**S3 location:** `s3a://openalex-ingest/awards/bhf/bhf_projects.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bhf_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/bhf/bhf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.bhf_raw;

## Step 1.5: Inspect raw + money/currency scan

Per runbook §1.5, scan for money-flavored columns even though the script
already parsed amounts from the PDFs (catches drift if the PDF layout changes).

Known source columns (all STRING per runbook §1.2.5): `funder_award_id`,
`reference_number`, `report_period`, `report_year_start`, `funder_scheme`,
`title`, `amount`, `currency`, `duration_months`, `lead_full_name`,
`lead_given_name`, `lead_family_name`, `institution`, `funding_type`,
`landing_page_url`, `source_pdf_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.bhf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.bhf_raw LIMIT 5;

In [ ]:
%sql
-- Money-shape scan per runbook §1.5: amount already parsed; confirm GBP range.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total
FROM openalex.awards.bhf_raw;

In [ ]:
%sql
-- Currency present (should be only 'GBP' or NULL).
SELECT currency, COUNT(*) AS rows FROM openalex.awards.bhf_raw GROUP BY currency;

## Step 1.6: Fail-fast — verify BHF funder row exists (Path A, F4320*)

The Step 2 transform `CROSS JOIN`s against `openalex.common.funder`. BHF is a
Crossref-registered F4320* funder, so this **must return exactly 1 row**. If 0,
STOP — the funder is unexpectedly missing from the dim.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320319992;  -- British Heart Foundation

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.bhf_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319992  -- British Heart Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(b.funder_award_id)
    ))) % 9000000000 AS id,
    b.title AS display_name,
    CAST(NULL AS STRING) AS description,           -- BHF publishes no abstracts
    f.funder_id,
    b.funder_award_id,
    TRY_CAST(b.amount AS DOUBLE) AS amount,
    b.currency,                                    -- 'GBP' from script, NULL if no amount
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    b.funding_type,                                -- research / fellowship / training
    b.funder_scheme,                               -- committee/scheme heading
    'bhf_annual_reports' AS provenance,
    CAST(NULL AS DATE) AS start_date,              -- no exact award date published
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(b.report_year_start AS INT) AS start_year,   -- FY the award was made
    CASE
        WHEN TRY_CAST(b.report_year_start AS INT) IS NOT NULL
         AND TRY_CAST(b.duration_months AS DOUBLE) IS NOT NULL
        THEN TRY_CAST(b.report_year_start AS INT)
             + CAST(CEIL(TRY_CAST(b.duration_months AS DOUBLE) / 12.0) AS INT)
        ELSE NULL
    END AS end_year,
    CASE
        WHEN b.lead_family_name IS NULL OR b.lead_family_name = '' THEN NULL
        ELSE struct(
            b.lead_given_name AS given_name,
            b.lead_family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,         -- BHF publishes no ORCIDs
            CAST(NULL AS DATE) AS role_start,
            struct(
                b.institution AS name,
                CASE WHEN b.funder_scheme ILIKE '%International%'
                     THEN CAST(NULL AS STRING) ELSE 'GB' END AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING,
        ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) AS investigators,
    b.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(b.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.bhf_raw b
CROSS JOIN funder_resolved f
WHERE b.funder_award_id IS NOT NULL
  AND b.title IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 395

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bhf_annual_reports' AND priority = 395;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    395 AS priority  -- BHF priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.bhf_awards;

## Step 6: Verification

Full §6.1–6.8 verification. BHF amount-coverage is NOT waived — expect
~99% `pct_amount`, single currency `'GBP'`, avg in the £100K–£1M range.

In [ ]:
%sql
SELECT COUNT(*) AS total_bhf_award_rows FROM openalex.awards.bhf_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.bhf_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(amount) AS has_amount,
    COUNT(start_year) AS has_start_year,
    COUNT(lead_investigator) AS has_pi,
    COUNT(funder_scheme) AS has_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_pi
FROM openalex.awards.bhf_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast check (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount, AVG(amount) AS avg_amount
FROM openalex.awards.bhf_awards;

In [ ]:
%sql
-- §6.4 sample inspection
SELECT id, SUBSTRING(display_name, 1, 60) AS title, funder_scheme, funding_type,
       amount, currency, start_year, end_year,
       lead_investigator.given_name AS given, lead_investigator.family_name AS family,
       lead_investigator.affiliation.name AS institution
FROM openalex.awards.bhf_awards
ORDER BY start_year DESC NULLS LAST, amount DESC NULLS LAST
LIMIT 12;

In [ ]:
%sql
-- §6.4a PI frequency check — top family names must be a real long-tail, never
-- an institution word. Repeat BHF Chairs (Bennett, Baker, ...) appear a few times.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.bhf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Scheme distribution + funding_type split
SELECT funder_scheme, funding_type, COUNT(*) AS rows, ROUND(SUM(amount)/1e6, 1) AS total_gbp_m
FROM openalex.awards.bhf_awards
GROUP BY funder_scheme, funding_type ORDER BY rows DESC LIMIT 25;

In [ ]:
%sql
-- §6.6 year distribution — expect ~2004–2025, ~120–260 awards/yr
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.bhf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- §6.5 funder consistency — should be exactly British Heart Foundation
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.bhf_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- §6.8 confirm rows reached the shared raw table at the right priority
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'bhf_annual_reports'
GROUP BY provenance, priority;